# Channel audit: M5 pilot in resumable shards

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arshia-HZ/mas-channel-audit/blob/main/notebooks/pilot.ipynb)

Runs **every arm** on the **200 pilot problems** with **2 replicates** for the primary
model (Qwen3-1.7B, temperature 0.7, 2048 tokens), in shards of about 50 problems. A
(subject, level) group is never split across shards, so the placebo arm always finds a
donor. Each finished shard is validated and **uploaded to the private dataset**, so a
disconnect costs at most the shard in progress.

**Resume:** just run all again. Finished shards are skipped without loading the model, an
interrupted shard continues from its saved generations, and a shard that finished but did
not upload is only uploaded. If the Google Drive folder is lost, shards are restored from
the dataset.

**Upload needs a Hugging Face token with write access** to the dataset (secret `HF_TOKEN`).
Set `UPLOAD = False` to keep everything on Drive only. Each shard takes roughly an hour and
a half to two hours on a T4; set `MAX_NEW_SHARDS` to stop a session after that many.

When all shards are done the notebook prints gates G1 to G3, the arm table and the
paired-difference SDs for the power calculation, and prepares the 40 failure messages for
you to label (gate G4). The pilot is never pooled with confirmatory data.

## 0. Settings

In [ ]:
MODEL_CONFIG = "configs/models/qwen3-1.7b.yaml"
BACKEND = "vllm"
REPLICATES = 2
SHARD_SIZE = 50
MAX_NEW_SHARDS = None  # e.g. 2 to stop this session after two new shards
UPLOAD = True  # push each finished shard to the private dataset
HF_REPO = "Arshia-HZ/mas-channel-audit-raw"
REF = "main"  # branch, tag or commit of the repository to run
REPO_URL = "https://github.com/Arshia-HZ/mas-channel-audit.git"
USE_DRIVE = True  # Colab only: keep shards in Google Drive

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB:
    REPO = "/content/mas-channel-audit"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/mas-channel-audit"
    else:
        RUN_ROOT = "/content/mas-channel-audit-runs"
elif IN_KAGGLE:
    REPO = "/kaggle/working/mas-channel-audit"
    RUN_ROOT = "/kaggle/working/runs"
else:  # a local checkout
    REPO = os.getcwd()
    if os.path.basename(REPO) == "notebooks":
        REPO = os.path.dirname(REPO)
    RUN_ROOT = os.environ.get("MCA_RUN_ROOT", os.path.join(REPO, "runs"))
os.makedirs(RUN_ROOT, exist_ok=True)


def sh(*cmd, cwd=None, check=True):
    """Run a command, show its output in the cell, raise if it fails."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(proc.stdout, end="")
    print(proc.stderr, end="", file=sys.stderr)
    if check and proc.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {proc.returncode}")
    return proc


print("colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", "| repo:", REPO)
print("results go to:", RUN_ROOT)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    if not os.path.isdir(f"{REPO}/.git"):
        sh("git", "clone", "-q", REPO_URL, REPO)
    sh("git", "fetch", "-q", "--all", cwd=REPO)
    sh("git", "checkout", "-q", REF, cwd=REPO)
    if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO).returncode == 0:
        sh("git", "pull", "-q", "--ff-only", cwd=REPO)  # on a branch: take the latest
    extra = "vllm" if BACKEND == "vllm" else "hf"
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", f".[{extra},data]", cwd=REPO)

os.chdir(REPO)
sys.path.insert(
    0, f"{REPO}/src"
)  # a running kernel does not see a fresh editable install
sh("git", "log", "--oneline", "-1")

**Hugging Face token.** Needed to upload shards (write access) and for the dataset restore.
Store it as a Colab or Kaggle secret named `HF_TOKEN`. It is never printed.

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token and IN_COLAB:
    from google.colab import userdata

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
if not token and IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # the secret is simply not set
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "not set (fine for the Qwen models)")

In [ ]:
import shutil
from importlib import metadata

if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
for pkg in ("vllm", "torch", "transformers", "numpy", "math-verify", "pydantic"):
    try:
        print(f"{pkg:<14}{metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## 2. Model and pilot problems

In [ ]:
from channel_audit.models import load_model_config

model = load_model_config(f"{REPO}/{MODEL_CONFIG}")
if model.backend != BACKEND:
    model = model.model_copy(update={"backend": BACKEND})
slug = model.model_id.split("/")[-1].lower()
RUN = f"{RUN_ROOT}/pilot_{slug}_{BACKEND}"
PREFIX = f"pilot/{slug}"
print(model.model_id, "@", model.revision[:12], "|", model.backend, model.dtype)
print("max_tokens:", model.max_tokens, "| max_model_len:", model.max_model_len)
print("shards go to:", RUN, "| remote:", HF_REPO if UPLOAD else "(upload off)")

In [ ]:
from channel_audit.data import (
    MATH_REVISION,
    assign_splits,
    load_frozen,
    load_math_test,
    write_split,
)

SPLIT_DIR = os.environ.get("MCA_SPLIT_DIR", f"{REPO}/data/problems")
# Rebuilds the split from the pinned MATH revision and checks it against the committed
# manifest; refuses to continue if anything differs.
rows, dropped = load_math_test()
manifest = write_split(
    assign_splits(rows), SPLIT_DIR, n_rows=len(rows) + len(dropped), dropped=dropped
)
problems = load_frozen(SPLIT_DIR, "pilot")
print("MATH revision:", MATH_REVISION)
print(
    "pilot split:",
    len(problems),
    "problems, sha256",
    manifest["files"]["pilot"]["sha256"][:16],
)

## 3. Run the shards

The model is loaded **once**, on the first shard that needs it.

In [ ]:
from channel_audit.runner import backend_for_run, config_for_model
from channel_audit.shards import (
    LazyBackend,
    hf_downloader,
    hf_uploader,
    make_shards,
    run_shards,
)

cfg = config_for_model(model, replicates=REPLICATES, log_prompts=True)
uploader = restorer = None
if UPLOAD:
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("UPLOAD needs the HF_TOKEN secret (write access)")
    uploader = hf_uploader(HF_REPO, token)
    restorer = hf_downloader(HF_REPO, token)

n_shards = len(make_shards(problems, SHARD_SIZE))
print(f"{len(problems)} problems in {n_shards} shards of about {SHARD_SIZE}")
results = run_shards(
    RUN,
    cfg,
    problems,
    LazyBackend(lambda: backend_for_run(cfg, problems)),
    size=SHARD_SIZE,
    uploader=uploader,
    restorer=restorer,
    remote_prefix=PREFIX,
    max_new_shards=MAX_NEW_SHARDS,
)

In [ ]:
from pathlib import Path

shard_dirs = sorted(
    d for d in Path(RUN).glob("shard_*") if (d / "SHARD_DONE.json").exists()
)
for r in results:
    print(
        f"{r.name}: {r.status}, {r.n_generations} generations, "
        f"G3 {'pass' if r.g3_ok else 'FAIL'}, uploaded {r.uploaded}"
    )
ALL_DONE = len(shard_dirs) == n_shards
print(
    f"{len(shard_dirs)} of {n_shards} shards finished",
    "- run all again to continue" if not ALL_DONE else "",
)

## 4. Gates and power inputs

Printed once every shard is done. G4 needs the 40 failure messages labelled by hand (next
section), so it shows as pending here.

In [ ]:
import json

from channel_audit.gates import format_pilot_report, pilot_report
from channel_audit.runner import open_run

if ALL_DONE:
    confirm_available = manifest["counts"]["confirm"]
    ctxs = [open_run(d) for d in shard_dirs]
    report = pilot_report(ctxs, confirm_available)
    print(format_pilot_report(report, None))
    with open(f"{RUN}/pilot_report.json", "w", encoding="utf-8") as fh:
        json.dump(report, fh, indent=1, sort_keys=True)
else:
    print("not all shards are finished yet")

## 5. Failure messages to label (gate G4)

Draws 40 failure messages at random (no use of correctness) and writes
`failure_sheet.md` and a blank `labels.json` into `g4/`. Read the sheet, fill each label
with `useful`, `vague` or `wrong`, and send me both files.

In [ ]:
from channel_audit.failure_review import (
    format_label_sheet,
    sample_failure_messages,
    write_label_template,
)

if ALL_DONE:
    G4_DIR = Path(RUN) / "g4"
    G4_DIR.mkdir(exist_ok=True)
    items = sample_failure_messages(ctxs, n=40, seed=0)
    sheet = G4_DIR / "failure_sheet.md"
    sheet.write_text(format_label_sheet(items), encoding="utf-8")
    if not (G4_DIR / "labels.json").exists():
        write_label_template(items, G4_DIR / "labels.json")
    print(f"{len(items)} messages in {sheet}")
    if IN_COLAB:
        from google.colab import files

        files.download(str(sheet))
        files.download(str(G4_DIR / "labels.json"))
        files.download(f"{RUN}/pilot_report.json")

**Send back:** the printed report (gates, arm table, contrasts), `pilot_report.json`, and the
filled `labels.json`. The raw shards are already in the private dataset.